
**La primera celda carga las fuentes principales desde Unity Catalog:**
- Importa librerías necesarias.
- Lee la tabla de vuelos (`silver_vuelos_cancelaciones`), el historial meteorológico (`silver_clima_historico`) y el catálogo de aeropuertos (`silver_aeropuertos_coordenadas`).
- Imprime mensajes de confirmación y auditoría de carga.

In [0]:
# -----------------------------------------------------------------------------
# CAPA GOLD - CELDA 1: CARGA DE FUENTES DESDE UNITY CATALOG
# -----------------------------------------------------------------------------
import pyspark.sql.functions as F
from pyspark.sql.window import Window

print("--- INICIANDO CAPA GOLD: LECTURA DIRECTA DE DISCO DELTA ---")

# 1. Conexión al universo analítico de vuelos (8.6 millones de filas)
df_vuelos = spark.read.table("workspace.default.silver_vuelos_cancelaciones")

# 2. Conexión al historial meteorológico diario de Open-Meteo (147,128 filas)
df_clima = spark.read.table("workspace.default.silver_clima_historico")

# 3. Conexión al catálogo geográfico persistido que estabilizamos en el notebook bronce_a_silver
df_aeropuertos_coordenadas = spark.read.table("workspace.default.silver_aeropuertos_coordenadas")

print(f" Vuelos Silver vinculados con éxito.")
print(f" Historial de clima Silver vinculado con éxito.")
print(f" Catálogo geoespacial Silver cargado con éxito ({df_aeropuertos_coordenadas.count()} nodos).")

**Identificamos los aeropuertos sin datos climáticos ("fantasmas") y, usando KDTree, asignamos a cada uno su aeropuerto vecino más cercano con clima disponible.**

In [0]:
# ------------------------------------------------------
# CAPA GOLD - CELDA 2: INDEXACIÓN ESPACIAL CON KDTREE DE SCIPY
# ------------------------------------------------------
from scipy.spatial import KDTree
import pandas as pd

print("--- CONSTRUYENDO ÁRBOL ESPACIAL DE VECINOS CERCANOS ---")

# 1. Identificamos en el clúster cuáles aeropuertos tienen clima y cuáles están vacíos
df_analisis_nulos = df_clima.groupBy("AirportCode").agg(F.count("MaxTemperature").alias("Datos_Validos"))
lista_fantasmas = [row["AirportCode"] for row in df_analisis_nulos.filter(F.col("Datos_Validos") == 0).select("AirportCode").collect()]

# 2. Convertimos el catálogo compacto a Pandas en el nodo maestro (347 filas)
df_geo_pandas = df_aeropuertos_coordenadas.toPandas()

df_buenos_geo = df_geo_pandas[~df_geo_pandas['Origin'].isin(lista_fantasmas)].reset_index(drop=True)
df_fantasmas_geo = df_geo_pandas[df_geo_pandas['Origin'].isin(lista_fantasmas)].reset_index(drop=True)

# 3. Entrenamos el Árbol KD con las coordenadas de los aeropuertos meteorológicamente estables
coordenadas_buenas = df_buenos_geo[['Latitude', 'Longitude']].values
arbol_espacial = KDTree(coordenadas_buenas)

# 4. Consultamos el árbol para mapear el "vecino salvador" (k=1) de cada uno de los 30 huérfanos
mapeo_vecinos_dict = {}
for idx, row in df_fantasmas_geo.iterrows():
    coord_fantasma = [row['Latitude'], row['Longitude']]
    distancia, indice_vecino = arbol_espacial.query(coord_fantasma, k=1)
    
    codigo_fantasma = row['Origin']
    codigo_vecino_salvador = df_buenos_geo.loc[indice_vecino, 'Origin']
    mapeo_vecinos_dict[codigo_fantasma] = codigo_vecino_salvador

print(f" Árbol KD consolidado con éxito.")
print(f"Mapeo de rescate generado para los {len(mapeo_vecinos_dict)} aeropuertos fantasmas:")
print(mapeo_vecinos_dict)

**Esta celda le inyecta el diccionario de SciPy a Spark Connect. Si un vuelo salió de un aeropuerto fantasma, Spark sabrá dinámicamente a qué aeropuerto clonarle el clima para ese día en específico. Al final, corremos una auditoría de calidad para verificar que salvamos el 100% de los vuelos.**

In [0]:
# ------------------------------------------------------------------
# CAPA GOLD - CELDA 3: REASIGNACIÓN GEOGRÁFICA Y LEFT JOIN MASIVO
# ------------------------------------------------------------------

# 1. Traducimos el diccionario de SciPy a una expresión lógica condicional de Spark
expresion_codigo_clima = F.col("ORIGIN")

for fantasma, vecino in mapeo_vecinos_dict.items():
    expresion_codigo_clima = (
        F.when(F.col("ORIGIN") == fantasma, vecino)
         .otherwise(expresion_codigo_clima)
    )

# 2. Creamos la columna de cruce adaptativa
df_vuelos_preparados = df_vuelos.withColumn(
    "Clima_Airport_Code",
    expresion_codigo_clima
)

print("--- EJECUTANDO LEFT JOIN MASIVO CON COLUMNAS COMPROBADAS ---")

# 3. JOIN conservando todas las columnas de vuelos y SOLO las meteorológicas
df_gold_completo = (
    df_vuelos_preparados.alias("v")
    .join(
        df_clima.alias("c"),
        (F.col("v.FlightDate") == F.col("c.WeatherDate")) &
        (F.col("v.Clima_Airport_Code") == F.col("c.AirportCode")),
        "left"
    )
    .select(
        "v.*",
        "c.MaxTemperature",
        "c.MinTemperature",
        "c.Precipitation",
        "c.Snowfall",
        "c.MaxWindSpeed"
    )
)

# Auditoría
total_vuelos_final = df_gold_completo.count()
nulos_finales = df_gold_completo.filter(
    F.col("MaxTemperature").isNull()
).count()

print("\n--- AUDITORÍA DE CALIDAD CAPA GOLD ---")
print(f"Total de vuelos preservados: {total_vuelos_final:,}")
print(f"Nulos meteorológicos residuales: {nulos_finales:,}")

**Se aplica funciones de ventana (Window) distribuidas para arrastrar los datos en el tiempo y sepultar ese 7.73% de nulos de forma definitiva
Asignar temperatura del día cercano **

In [0]:
# -----------------------------------------------------------------------------
# CAPA GOLD - CELDA 4: IMPUTACIÓN POR VENTANA TEMPORAL (FORWARD / BACKWARD FILL)
# ------------------------------------------------------------------------------
from pyspark.sql.window import Window
import pyspark.sql.functions as F

print("--- INICIANDO RELLENO POR CONTINUIDAD TEMPORAL (CAMINO 2) ---")

# 1. Definimos la ventana analítica: Particionamos por Aeropuerto Origen y ordenamos cronológicamente
ventana_temporal = Window.partitionBy("ORIGIN").orderBy("FlightDate")

# 2. Aplicamos 'last' con ignorenulls=True. 
# Esto busca hacia atrás en el tiempo el último día con clima válido en ese aeropuerto (Forward Fill)
df_gold_forward = df_gold_completo \
    .withColumn("MaxTemperature", F.last("MaxTemperature", True).over(ventana_temporal)) \
    .withColumn("MinTemperature", F.last("MinTemperature", True).over(ventana_temporal)) \
    .withColumn("Precipitation", F.last("Precipitation", True).over(ventana_temporal)) \
    .withColumn("Snowfall", F.last("Snowfall", True).over(ventana_temporal)) \
    .withColumn("MaxWindSpeed", F.last("MaxWindSpeed", True).over(ventana_temporal))

# 3. Respaldo por si el bache ocurrió exactamente el primer día del año (1 de enero)
# Si no hay día anterior, busca el clima del día siguiente (Backward Fill)
ventana_inversa = Window.partitionBy("ORIGIN").orderBy(F.col("FlightDate").desc())

df_gold_final = df_gold_forward \
    .withColumn("MaxTemperature", F.last("MaxTemperature", True).over(ventana_inversa)) \
    .withColumn("MinTemperature", F.last("MinTemperature", True).over(ventana_inversa)) \
    .withColumn("Precipitation", F.last("Precipitation", True).over(ventana_inversa)) \
    .withColumn("Snowfall", F.last("Snowfall", True).over(ventana_inversa)) \
    .withColumn("MaxWindSpeed", F.last("MaxWindSpeed", True).over(ventana_inversa))

print("--- AUDITORÍA FINAL POST-IMPUTACIÓN TEMPORAL ---")
total_filas = df_gold_final.count()
nulos_absolutos = df_gold_final.filter(F.col("MaxTemperature").isNull()).count()

print(f" Total de vuelos preservados intactos: {total_filas}")
print(f" Nulos climáticos remanentes en el dataset: {nulos_absolutos}")

if nulos_absolutos == 0:
    print(" 0% nulos de forma orgánica. Dataset está blindado y denso para la red neuronal.")

**Limpieza de nulos que no pudieron ser rescatados**

In [0]:
# --------------------------------------------------------------------------
# CAPA GOLD - CELDA 5: LIMPIEZA DE FILAS CON NULOS NO RESCATADOS
# ---------------------------------------------------------------------------
import pyspark.sql.functions as F

print("--- EJECUTANDO LIMPIEZA DE FILAS CON NULOS RESIDUALES ---")

# Como las columnas climáticas son las únicas duplicadas y ya las controlaste,
# podés usar F.col() plano sin miedo a colisiones.
df_gold_eficiente = df_gold_final.filter(
    (F.col("MaxTemperature").isNotNull()) & 
    (F.col("Month").isNotNull()) & 
    (F.col("DayOfWeek").isNotNull()) & 
    (F.col("DayOfMonth").isNotNull())
)

print(f" Volumen neto tras la limpieza: {df_gold_eficiente.count():,} vuelos.")



**La celda 6 realiza dos tareas principales:**
- Aplica transformaciones cíclicas a las variables temporales (`Month`, `DayOfWeek`, `DayOfMonth`) para que los modelos de machine learning puedan capturar la naturaleza periódica del tiempo.
- Escala las variables meteorológicas (`MaxTemperature`, `MinTemperature`, `Precipitation`, `Snowfall`, `MaxWindSpeed`) a un rango normalizado, facilitando el entrenamiento de modelos y mejorando la convergencia.

In [0]:

# ------------------------------------------------------------------------------
# CAPA GOLD - CELDA 6: (Transformación Cíclica y Escalado)
# ------------------------------------------------------------------------------
import pyspark.sql.functions as F
import math

# 1. Transformación Cíclica leyendo del paso anterior (df_gold_eficiente)
df_cyclic = df_gold_eficiente \
    .withColumn("sin_month", F.sin(2 * math.pi * F.col("Month") / 12.0)) \
    .withColumn("cos_month", F.cos(2 * math.pi * F.col("Month") / 12.0)) \
    .withColumn("sin_dayofweek", F.sin(2 * math.pi * F.col("DayOfWeek") / 7.0)) \
    .withColumn("cos_dayofweek", F.cos(2 * math.pi * F.col("DayOfWeek") / 7.0)) \
    .withColumn("sin_dayofmonth", F.sin(2 * math.pi * F.col("DayOfMonth") / 31.0)) \
    .withColumn("cos_dayofmonth", F.cos(2 * math.pi * F.col("DayOfMonth") / 31.0))

print(" Columnas de tiempo desambiguadas y convertidas a formato cíclico.")

# 2. Escalado Min-Max de Variables Climáticas
df_gold_features = df_cyclic \
    .withColumn("scaled_max_temp", (F.col("MaxTemperature") - (-30.0)) / (50.0 - (-30.0))) \
    .withColumn("scaled_min_temp", (F.col("MinTemperature") - (-40.0)) / (40.0 - (-40.0))) \
    .withColumn("scaled_precipitation", F.when(F.col("Precipitation") < 0, 0).otherwise(F.col("Precipitation") / 150.0)) \
    .withColumn("scaled_snowfall", F.col("Snowfall") / 50.0) \
    .withColumn("scaled_wind_speed", F.col("MaxWindSpeed") / 120.0)

print(" Variables meteorológicas escaladas correctamente.")



In [0]:
# --------------------------------------------------------------------------
# CAPA GOLD - Continuación celda 6: INGENIERÍA DE CARACTERÍSTICAS DE HORA 
#-----------------------------------------------------------------------------
import pyspark.sql.functions as F
import math

# Extraemos la hora (ej: 1830 -> 18)
df_gold_features = df_gold_features.withColumn("hour", (F.col("CRSDepTime") / 100).cast("int"))

# Aplicamos codificación cíclica nativa en Spark
df_gold_features = df_gold_features \
    .withColumn("sin_hour", F.sin(2 * math.pi * F.col("hour") / 24.0)) \
    .withColumn("cos_hour", F.cos(2 * math.pi * F.col("hour") / 24.0))


**La celda 6.5 realiza ingeniería de características logísticas de red:**
- Crea bloques horarios para salidas y llegadas, agrupando vuelos por hora en vez de minuto exacto.
- Calcula el volumen de vuelos por bloque horario y aeropuerto de origen/destino (`origin_hourly_volume`, `dest_hourly_volume`) usando funciones de ventana.
- Ajusta la fecha de llegada si el vuelo cruza de día.
- Aplica logaritmo y escalado Min-Max a los volúmenes para normalizarlos y suavizar diferencias entre aeropuertos grandes y pequeños.
- El resultado son variables enriquecidas que reflejan la congestión y dinámica operativa de la red en cada bloque horario.

In [0]:
# ------------------------------------------------------------------------------
# CAPA GOLD - CELDA 6.5: INGENIERÍA DE CARACTERÍSTICAS LOGÍSTICAS DE RED
# CAMBIO NUEVO:
# - Congestión por bloque horario real, no por minuto exacto.
# - Escalado Min-Max de volumen usando datos ya existentes.
# ------------------------------------------------------------------------------

from pyspark.sql.window import Window
import pyspark.sql.functions as F
import math

print("--- CALCULANDO CONGESTIÓN POR BLOQUES HORARIOS ---")

# ----------------------------------------------------------------------
# NUEVO 1: Crear bloques horarios programados.
# Ejemplos:
# 0815 -> 8
# 0930 -> 9
# 0005 -> 0
# 2400 -> 0
# ----------------------------------------------------------------------

df_gold_con_horas = (
    df_gold_features
    .withColumn(
        "dep_hour_block",
        F.when(F.col("CRSDepTime") == 2400, 0)
         .otherwise(F.floor(F.col("CRSDepTime") / 100))
         .cast("int")
    )
    .withColumn(
        "arr_hour_block",
        F.when(F.col("CRSArrTime") == 2400, 0)
         .otherwise(F.floor(F.col("CRSArrTime") / 100))
         .cast("int")
    )
)

# ----------------------------------------------------------------------
# NUEVO 2: Fecha estimada de llegada.
# Si la hora programada de llegada es menor que la de salida, se asume
# que la llegada ocurre al día siguiente.
# ----------------------------------------------------------------------

df_gold_con_horas = df_gold_con_horas.withColumn(
    "ScheduledArrivalDate",
    F.date_add(
        F.col("FlightDate"),
        F.when(
            F.col("CRSArrTime") < F.col("CRSDepTime"),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)

# ----------------------------------------------------------------------
# NUEVO 3: Ventanas por aeropuerto, fecha y bloque de una hora.
# Antes se usaba CRSDepTime exacto; por ejemplo 08:00 y 08:15 se
# consideraban grupos distintos. Ahora pertenecen al mismo bloque.
# ----------------------------------------------------------------------

ventana_origen_saturacion = Window.partitionBy(
    "Origin",
    "FlightDate",
    "dep_hour_block"
)

ventana_destino_saturacion = Window.partitionBy(
    "Dest",
    "ScheduledArrivalDate",
    "arr_hour_block"
)

df_gold_con_volumen = (
    df_gold_con_horas
    .withColumn(
        "origin_hourly_volume",
        F.count(F.lit(1)).over(ventana_origen_saturacion)
    )
    .withColumn(
        "dest_hourly_volume",
        F.count(F.lit(1)).over(ventana_destino_saturacion)
    )
)

# ----------------------------------------------------------------------
# NUEVO 4: Aplicamos logaritmo para suavizar aeropuertos muy grandes.
# Después usamos Min-Max para que los volúmenes queden entre 0 y 1.
# ----------------------------------------------------------------------

df_volumen_log = (
    df_gold_con_volumen
    .withColumn(
        "origin_log_volume",
        F.log1p(F.col("origin_hourly_volume").cast("double"))
    )
    .withColumn(
        "dest_log_volume",
        F.log1p(F.col("dest_hourly_volume").cast("double"))
    )
)

estadisticas_volumen = df_volumen_log.agg(
    F.min("origin_log_volume").alias("origin_min"),
    F.max("origin_log_volume").alias("origin_max"),
    F.min("dest_log_volume").alias("dest_min"),
    F.max("dest_log_volume").alias("dest_max")
).first()

origin_min = float(estadisticas_volumen["origin_min"])
origin_max = float(estadisticas_volumen["origin_max"])
dest_min = float(estadisticas_volumen["dest_min"])
dest_max = float(estadisticas_volumen["dest_max"])

def escalar_min_max(nombre_columna, valor_minimo, valor_maximo):
    if valor_maximo == valor_minimo:
        return F.lit(0.0)

    return (
        (F.col(nombre_columna) - F.lit(valor_minimo)) /
        F.lit(valor_maximo - valor_minimo)
    )

df_gold_v2 = (
    df_volumen_log
    .withColumn(
        "scaled_origin_volume",
        escalar_min_max(
            "origin_log_volume",
            origin_min,
            origin_max
        )
    )
    .withColumn(
        "scaled_dest_volume",
        escalar_min_max(
            "dest_log_volume",
            dest_min,
            dest_max
        )
    )
    .drop(
        "origin_log_volume",
        "dest_log_volume"
    )
)

print("¡Características de congestión por bloque horario creadas con éxito!")
print(f"Rango volumen origen: {origin_min:.4f} a {origin_max:.4f}")
print(f"Rango volumen destino: {dest_min:.4f} a {dest_max:.4f}")

**Ahora esa celda 7 consolida el dataset final y lo guarda en Unity Catalog como `gold_vuelos_features`. **

In [0]:
# ------------------------------------------------------------------------------
# CAPA GOLD - CELDA 7: CONSOLIDACIÓN DE TABLA FINAL
# CAMBIO NUEVO:
# - Agrega Reporting_Airline como variable predictora.
# - Mantiene ORIGIN y DEST en mayúscula para el notebook de ML.
# - overwriteSchema=true actualiza el esquema Delta existente.
# ------------------------------------------------------------------------------

import pyspark.sql.functions as F

target_gold_table = "workspace.default.gold_vuelos_features"

print(f"Guardando dataset analítico multiclase en {target_gold_table}...")

# Construcción del dataset final
df_final_to_disk = df_gold_v2.select(
    "FlightDate",

    # nombres consistentes con el notebook ML
    F.col("Origin").alias("ORIGIN"),
    F.col("Dest").alias("DEST"),

    # aerolínea incluida como feature para ML
    F.coalesce(
        F.col("Reporting_Airline"),
        F.lit("UNKNOWN")
    ).alias("Reporting_Airline"),

    "sin_month",
    "cos_month",
    "sin_dayofweek",
    "cos_dayofweek",
    "sin_dayofmonth",
    "cos_dayofmonth",
    "scaled_max_temp",
    "scaled_min_temp",
    "scaled_precipitation",
    "scaled_snowfall",
    "scaled_wind_speed",
    "CRSElapsedTime",
    "Distance",
    "scaled_origin_volume",
    "scaled_dest_volume",
    "sin_hour",
    "cos_hour",

    # Se mantiene lógica original de etiquetas
    F.when(F.col("Cancelled") == 1.0, 2.0)
     .when(F.col("DepDelay") > 15.0, 1.0)
     .otherwise(0.0)
     .cast("double")
     .alias("label")
)
# overwriteSchema permite guardar la nueva columna Reporting_Airline
# aunque la tabla gold_vuelos_features ya existiera con el esquema anterior.
(
    df_final_to_disk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_gold_table)
)

print("\n¡PROCESO GOLD CONCLUIDO CON ÉXITO!")
print(f"La tabla '{target_gold_table}' quedó disponible.")